# K-Means Clustering Analysis: Census Income Dataset

This notebook performs a K-Means clustering analysis on the Census Income dataset.
All reusable logic (loading, preprocessing, modeling) lives in `k_means_clustering.py` and is imported here.


## 1. Objectives

- Load and inspect the Census Income data
- Select and preprocess features for clustering
- Use the elbow method to choose an appropriate number of clusters
- Fit a K-Means model and interpret cluster assignments
- Visualize clusters using PCA
- Summarize and interpret cluster profiles


In [8]:
import sys
print(sys.executable)

c:\Users\jimen\Documents\INDE_577_Final_Project\INDE-577-Final-Project\.venv-1\Scripts\python.exe


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

ModuleNotFoundError: No module named 'pandas'

In [ ]:
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..','src')))

from rice_ml.unsupervised_learning.k_means_clustering import (
    load_census_data,
    clean_census_data,
    encode_features,
    scale_features,
    compute_elbow_inertia,
    fit_kmeans,
    run_pca,
    attach_clusters,
    summarize_clusters,
)

ModuleNotFoundError: No module named 'pandas'

## 3. Load and Inspect the Data

Update the path below if your CSV is located somewhere else in your repository.


In [ ]:
data_path = "datasets/census_income.csv"  

# Load the dataset
df = load_census_data(data_path)

df.head()  # quick preview


In [ ]:
# Basic info about the dataset
df.info()


## 4. Feature Selection and Preprocessing

We choose a subset of features that are likely related to income, including both numerical and categorical variables.
We then:

1. Keep only the selected columns
2. Replace `'?'` with missing values and drop incomplete rows
3. One-hot encode categorical variables
4. Standardize features for K-Means


In [ ]:
# Define the feature columns
numeric_cols = [
    "age",
    "education_num",
    "hours_per_week",
    "capital_gain",
    "capital_loss",
]

categorical_cols = [
    "occupation",
    "marital_status",
    "workclass",
    "sex",
]

selected_cols = numeric_cols + categorical_cols

# Clean and subset the data
df_clean = clean_census_data(df, selected_cols)
df_clean.head()


In [ ]:
# One-hot encode categorical variables
X_encoded = encode_features(df_clean, categorical_cols)
X_encoded.head()


In [ ]:
# Scale the features
X_scaled, scaler = scale_features(X_encoded)
X_scaled.shape


## 5. Elbow Method: Choosing the Number of Clusters

We compute the K-Means inertia (within-cluster sum of squares) for a range of K values to look for an "elbow" point.


In [ ]:
# Range of K values to try
k_values = range(2, 11)

elbow_results = compute_elbow_inertia(X_scaled, k_values)
elbow_results


In [ ]:
# Plot the elbow curve
ks = [k for k, _ in elbow_results]
inertias = [inertia for _, inertia in elbow_results]

plt.figure(figsize=(6, 4))
plt.plot(ks, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow Method for Optimal K")
plt.xticks(ks)
plt.grid(True)
plt.show()


Based on the elbow plot above, choose a value for **K** where the decrease in inertia starts to slow down.
Update `best_k` below with your chosen value.


In [ ]:
# 6. Fit the final K-Means model
best_k = 4  # <-- adjust this based on the elbow plot

kmeans_model = fit_kmeans(X_scaled, n_clusters=best_k)
labels = kmeans_model.labels_

# Attach cluster labels back to the cleaned data
df_clusters = attach_clusters(df_clean, labels, label_name="cluster")

# Cluster sizes
df_clusters["cluster"].value_counts().sort_index()


## 7. Visualize Clusters with PCA

We use Principal Component Analysis (PCA) to project the high-dimensional data to two dimensions for visualization.


In [ ]:
# Run PCA for 2D visualization
pca, X_pca = run_pca(X_scaled, n_components=2)

pc1 = X_pca[:, 0]
pc2 = X_pca[:, 1]

plt.figure(figsize=(7, 5))
scatter = plt.scatter(pc1, pc2, c=labels, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Census Income Clusters (PCA 2D Projection)")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.grid(True)
plt.show()


## 8. Cluster Profiles

Finally, we summarize key numeric features by cluster to understand how groups differ.


In [ ]:
cluster_summary = summarize_clusters(df_clusters, "cluster", numeric_cols)
cluster_summary


You can also inspect a few example rows from each cluster to get a more qualitative sense of the groups.


In [ ]:
# Example records from each cluster
for c in sorted(df_clusters["cluster"].unique()):
    display(df_clusters[df_clusters["cluster"] == c].head(5))
    print("-" * 80)
